# Triagem de Dengue — Pipeline reprodutível end-to-end

**Notebook canônico de reprodução** dos 40 experimentos + Wilcoxon pareado + avaliação final no teste.

Este notebook é a **versão executável ponta-a-ponta** da pipeline. Para EDA detalhada e exploração inicial dos dados, ver `recife_dengue_ldmc_jvlm2.ipynb`.

**Tempo total**: ~5min (apenas figuras a partir do MLflow já populado) até ~8-11h (rodar 40 experimentos do zero).

**Estrutura**:

1. Setup
2. Download do dataset (HuggingFace, idempotente)
3. Inspeção rápida do dado
4. Reprodução dos 40 experimentos (opcional — pular se MLflow já tem)
5. Geração de figuras + Wilcoxon + avaliação final
6. Inspeção dos resultados-chave
7. Próximos passos


## 1. Setup

Adiciona o root do projeto ao `sys.path` pra importar `src/` e `scripts/` daqui.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Detecta a raiz do projeto (notebook em notebooks/, raiz é o pai)
PROJECT_ROOT = Path(os.getcwd()).resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')
print(f'Python: {sys.version.split()[0]}')


## 2. Download do dataset

Idempotente: pula se `data/dataset_harmonizado.parquet` já existe.


In [ ]:
result = subprocess.run(
    [sys.executable, 'scripts/download_data.py'],
    capture_output=True, text=True, check=True,
)
print(result.stdout)


## 3. Inspeção rápida do dado

Carrega o treino (via `load_train`, sem tocar o teste) e mostra:
- Dimensões (linhas × features)
- Distribuição das 3 classes
- Nomes das colunas após cleanup


In [ ]:
from src.data_loader import load_train

X_train, y_train = load_train()
print(f'Treino: {X_train.shape[0]:,} linhas × {X_train.shape[1]} features')
print()
print('Distribuição das classes (treino):')
for cls, name in zip([0, 1, 2], ['Descartado', 'Comum', 'Alerta/Grave']):
    n = (y_train == cls).sum()
    pct = 100 * n / len(y_train)
    print(f'  {cls} {name:<14} {n:>6,} ({pct:.2f}%)')
print()
print(f'Features (n={X_train.shape[1]}):')
print('  ', list(X_train.columns))


## 4. Reprodução dos 40 experimentos (opcional)

⚠ **Esta célula leva ~8-11h locally** (40 runs sequenciais). Por padrão **está comentada** porque o repo já vem com os runs sincronizados (via `scripts/sync_mlflow_from_apuana.py`).

**Modos de uso**:
- **Demonstração rápida** (recomendado): pular esta célula. As análises na seção 5 leem o `mlflow_dengue.db` existente.
- **Reprodução completa**: descomente as células abaixo. Para acelerar use SLURM no Apuana (`scripts/apuana_run_variant.sh`) ou rode em subset (`--variant v2` ou `--algo lightgbm`).

**Estado atual do MLflow**:


In [ ]:
# Conferir quantos runs já estão no MLflow local
from mlflow.tracking import MlflowClient

MLFLOW_URI = f'sqlite:///{PROJECT_ROOT}/mlflow_dengue.db'
client = MlflowClient(tracking_uri=MLFLOW_URI)
exp = client.get_experiment_by_name('triagem-dengue')
if exp is None:
    print('❌ MLflow vazio — precisa rodar os experimentos (próxima célula).')
else:
    runs = client.search_runs([exp.experiment_id],
                              filter_string="tags.source = 'apuana'",
                              max_results=200)
    print(f'✓ {len(runs)} runs sincronizados no MLflow local.')
    if len(runs) >= 40:
        print('  Pronto pra rodar a seção 5 (análises). Pular célula seguinte.')
    else:
        print(f'  ⚠ Esperado 40 runs; achou {len(runs)}. Considere rodar experimentos.')


In [ ]:
# >>> DESCOMENTE pra reproduzir 40 experimentos do zero (~8-11h) <<<
# result = subprocess.run(['bash', 'scripts/run_all_experiments.sh'],
#                         capture_output=False, check=True)

# >>> OU subset (mais rápido pra demo) <<<
# result = subprocess.run(
#     ['bash', 'scripts/run_all_experiments.sh', '--algo', 'decision_tree'],
#     capture_output=False, check=True,
# )

print('(Célula desabilitada por padrão — descomente acima pra rodar.)')


## 5. Geração de figuras + Wilcoxon + avaliação final

Esta célula é **rápida (~5min)** e gera todos os 22 artifacts em `reports/figures/`:

- 4 grids de curvas treino vs validação (1 por variante)
- 4 grids de matrizes de confusão CV
- 4 tabelas + barplots por variante
- 4 scatters Pareto F1 × Recall Alerta/Grave
- 1 comparativo cross-variante (slide 12)
- Wilcoxon pareado 30 testes (Holm-Bonferroni)
- Avaliação final no teste (libera o sentinel `I_AM_IN_FINAL_EVALUATION`)


In [ ]:
result = subprocess.run(['bash', 'scripts/build_all_figures.sh'],
                        capture_output=True, text=True, check=True)
print(result.stdout[-3000:])  # últimos chars (resumo)


## 6. Inspeção dos resultados-chave

### 6.1 Wilcoxon pareado — pivô Δ F1 × variante


In [ ]:
import pandas as pd

df = pd.read_csv('reports/wilcoxon_paired.csv')
pivot = df.pivot(index='algoritmo', columns='variante', values='delta')
pivot = pivot.reindex(df['algoritmo'].drop_duplicates().tolist())  # ordem original
print('Δ F1-macro (variante − v1_baseline):')
print(pivot.round(4).to_string())
print()
print('Interpretação: positivo = variante > baseline.')


### 6.2 Comparação cross-variante (figura do slide 12)


In [ ]:
from IPython.display import Image, display
display(Image('reports/figures/cross_variant_comparison.png'))


### 6.3 Avaliação final no teste — matriz de confusão

Modelo final: `lightgbm + v2_smote` (escolhido por F1-macro CV = 0.4413 — mais alta entre as 40 combinações).


In [ ]:
display(Image('reports/figures/final_test_confusion_matrix.png'))


### 6.4 Avaliação final no teste — curvas ROC e PR


In [ ]:
display(Image('reports/figures/final_test_roc_pr.png'))


### 6.5 Classification report — modelo final


In [ ]:
with open('reports/figures/final_test_classification_report.txt') as f:
    print(f.read())


## 7. Próximos passos

- **Inspecionar runs no MLflow UI**: `mlflow ui --backend-store-uri sqlite:///mlflow_dengue.db`
- **Wilcoxon detalhado com effect sizes**: ver `reports/wilcoxon_paired_summary.md`
- **Limitações metodológicas (L1-L7)**: ver `README.md § Limitações`
- **EDA detalhada do dataset**: ver `notebooks/recife_dengue_ldmc_jvlm2.ipynb`

---

**Reprodutibilidade verificada**:
- `random_state = 42` em todos os splits/estimadores
- `StratifiedKFold(5, shuffle=True, random_state=42)` em todas buscas HP
- Sentinel `I_AM_IN_FINAL_EVALUATION` liberado uma única vez por execução (em `scripts/final_evaluation.py`)
